# 다음 포탈 크롤링 - HTTP 기반 정적 크롤링 

In [22]:
# 필요시 패키지 설치 (주석 해제 후 실행)
# !pip install beautifulsoup4
# !pip install lxml
# !pip install pandas
# !pip install tqdm

In [23]:
import logging
import random
import glob
import urllib.request
import urllib.error
import json
import socket
import threading
import collections
import urllib.parse
import http.cookiejar
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urlparse
from bs4 import BeautifulSoup as bs
from IPython.display import display
import time
import pandas as pd
import re
import warnings
from tqdm import tqdm
from IPython.display import display
import os
SAVE_PATH = "./data_csv/"
OVERLAP_PATH = "./overlap_data"
os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(OVERLAP_PATH, exist_ok=True)
warnings.filterwarnings(action="ignore")
# 로그 설정
logging.basicConfig(
level=logging.INFO,
format="%(asctime)s - %(levelname)s - %(message)s"
)
print("라이브러리 로드 완료")


라이브러리 로드 완료


## 1. 설정
### 1.1 키워드 및 수집 대상 설정
- 메인 키워드와 서브 키워드를 설정할 수 있어요
- 광고 관련 키워드를 포함한 게시물이 검색되지 않게 할 수 있어요
- 다음 포털 텍스트 수집 경로 5가지 중 원하는 요소를 수집할 수 있어요(기본값: 카페와 블로그)


In [33]:
# ==================== [상단 통합 설정] ====================
# 1. 수집 기간 설정 (YYYYMMDD 형식)
START_DATE = "20240101"  # 수집 시작일
END_DATE = "20260630"    # 수집 종료일

# 2. 키워드당 최대 수집 건수 제한
#    None이면 (수집 기간의 월 수 x 150)으로 자동 계산합니다.
#    (다음 검색은 한 구간당 최대 150개까지만 보여줍니다: 10페이지 x 15개)
#    숫자를 직접 넣으면 그 값으로 제한합니다.
TARGET_COUNT = None
MAX_RESULTS_PER_PERIOD = 150

# 2-1. 본문이 이 글자 수를 넘는 글은 수집하지 않고 블랙리스트에 올립니다. (None이면 사용 안 함 = 길이와 관계없이 모두 수집)
#      쓰려면 MAX_CONTENT_CHARS = 10000 처럼 숫자를 넣으세요.
MAX_CONTENT_CHARS = None

# 3. 동시 실행 워커 수 설정
MAX_WORKERS = 8

# 4. 검색어 단위 제외 키워드 설정
eliminate_ads_keyword = ""

# 5. 다음 포털 텍스트 수집 경로 및 대상 선택 (0: 카페, 1: 티스토리, 2: 브런치, 3: 웹문서, 4: 뉴스)
DAUM_TARGETS = {
    0: {"name": "다음카페", "w": "fusion", "col": "cafe"},
    1: {"name": "다음블로그_티스토리", "w": "fusion", "col": "tistory"},
    2: {"name": "다음블로그_브런치", "w": "fusion", "col": "brunch"},
    3: {"name": "다음웹문서", "w": "fusion", "col": "web"},
    4: {"name": "다음뉴스", "w": "news", "col": ""},
}
SELECTED_TARGETS = [0]

# 6. 메인 키워드 설정
main_keywords = ["귀농", "귀촌", "은퇴", "농부", "농사", "농촌"]

# 7. 세부 키워드 설정
invest_key = [
    # p1 (56개)
    "건강식", "아침", "운동", "산책", "부항", "함께운동", "제철음식", "식도염", "식사량", "뷰", "영양제", "아침루틴", "요리", "텃밭", "항산화", "위염", "역류성", "스트레스", "바이오리듬", "소화", "기온차", "일교차", "위건강", "해독주스", "식욕조절", "감기", "한약", "몸보신", "창문", "옆집", "소리", "냄새", "향기", "아침", "음악", "찬 음식", "계획", "환기", "공기", "빛", "소음", "온도", "등", "면역력", "시간", "보일러", "코골이", "암막", "이산화탄소", "피곤", "환절기", "으슬으슬", "노안", "자급자족", "혈당", "연당기",
    # p2 (51개)
    "새벽", "고령", "낙상", "문턱", "방법", "요추", "EMS패드", "병원", "혈압", "피검사", "고혈압", "고지혈", "신부전증", "천식", "위치", "약국", "간수치", "오한", "부정맥", "불면증", "멜라토닌", "마그네슘", "갱년기", "허리", "중성지방", "당뇨식", "심박수", "방광", "우울", "두통", "모니터링", "콜레스테롤", "뇌파", "당뇨", "자도자도", "곰팡이", "이동거리", "맥아리", "번아웃", "간병", "컨디션", "낮잠", "마사지", "재발", "폐경", "찜질", "반신욕", "명상", "명상", "수족냉증", "알레르기",
    # p3 (69개)
    "오도이촌", "라디에이터", "주말부부", "모임", "동호회", "초대", "홈파티", "전기세", "에너지", "촌캉스", "온도차", "위치", "야간", "응급", "알림", "해충", "짐", "작업", "바베큐", "주차", "음식", "나방", "배달", "음식물", "개미", "응급처치", "세스코", "밤산행", "교통", "불멍", "키트", "유통기한", "벌초", "예초기", "아지트", "반찬", "김장", "가구", "가전", "악취", "냉기", "물온도", "청소", "타일", "곰팡이", "동물병원", "CCTV", "경비", "알람", "빨래", "건조", "각방", "간섭", "독립", "혼밥", "택배", "빈집 관리", "동파", "누수", "세탁", "말리기", "청소", "사생활", "미루기", "단열", "잡초", "제초", "잔디", "풀",
    # p4 (57개)
    "먼지", "자전거", "낚시", "등산", "소독", "반려동물", "담금주", "등산", "음악", "자전거", "수영", "공부", "악기", "바둑", "커피", "와인", "가드닝", "목공", "소음", "스피커", "노래", "텃밭", "운동", "폼롤러", "중노동", "관절염", "삐끗", "과수원", "농번기", "부추", "무더위", "보드카", "인테리어", "무릎통증", "허리디스크", "허리통증", "엘보", "손가락", "유기농", "막걸리", "농한기", "씨앗", "작황", "옥수수", "화단", "채소", "방울토마토", "밤줍기", "수확", "제초", "정경", "조립식", "담소", "데크", "쉼터", "돌밭", "농막",
    # p5 (47개)
    "탈모", "피부", "손등", "자외선", "발진", "리프팅", "햇빛", "땀", "흙먼지", "세안", "보습", "야외활동", "주름", "마사지", "턱선", "녹물", "필터", "알러지", "진정", "재생", "자외선", "건조한 환경", "홈케어", "스킨케어", "피부관리", "근력", "체중관리", "피부케어", "기미", "검버섯", "노화", "안티에이징", "디바이스", "뷰티", "1일 1팩", "미백", "칙칙", "내장지방", "통통", "재발", "트러블", "화상", "붓기", "부음", "허리디스크", "혈액순환", "눈부심"
]

def generate_monthly_ranges(start_date_str, end_date_str):
    """시작일과 종료일을 받아 월별 (시작일, 종료일) 구간을 최신순으로 자동 분할 생성"""
    from datetime import datetime
    import calendar
    start_dt = datetime.strptime(start_date_str, "%Y%m%d")
    end_dt = datetime.strptime(end_date_str, "%Y%m%d")
    ranges = []
    curr_year, curr_month = end_dt.year, end_dt.month
    start_year, start_month = start_dt.year, start_dt.month
    while (curr_year, curr_month) >= (start_year, start_month):
        _, last_day = calendar.monthrange(curr_year, curr_month)
        if curr_year == start_year and curr_month == start_month:
            m_start = start_dt.strftime("%Y%m%d")
        else:
            m_start = f"{curr_year:04d}{curr_month:02d}01"
        if curr_year == end_dt.year and curr_month == end_dt.month:
            m_end = end_dt.strftime("%Y%m%d")
        else:
            m_end = f"{curr_year:04d}{curr_month:02d}{last_day:02d}"
        ranges.append((m_start, m_end))
        if curr_month == 1:
            curr_month = 12
            curr_year -= 1
        else:
            curr_month -= 1
    return ranges

YEAR_RANGES = generate_monthly_ranges(START_DATE, END_DATE)

_auto_target = TARGET_COUNT is None
if _auto_target:
    TARGET_COUNT = len(YEAR_RANGES) * MAX_RESULTS_PER_PERIOD

print("========== 선택된 수집 대상 ==========")
for target_idx in SELECTED_TARGETS:
    if target_idx in DAUM_TARGETS:
        target_info = DAUM_TARGETS[target_idx]
        print(f"번호 [{target_idx}] -> {target_info['name']} 수집 대상으로 지정되었습니다.")
    else:
        print(f"번호 [{target_idx}] -> 유효하지 않은 번호입니다. (선택 가능 번호: {list(DAUM_TARGETS.keys())})")
print(f"수집 기간 설정: {START_DATE} ~ {END_DATE} (총 {len(YEAR_RANGES)}개 월별 구간 자동 생성)")
print(f"키워드당 최대 수집 상한: {TARGET_COUNT}개" + (f" (자동: {len(YEAR_RANGES)}개월 x {MAX_RESULTS_PER_PERIOD}개)" if _auto_target else "") + f" | 동시 실행 워커 수: {MAX_WORKERS}개")


========== 선택된 수집 대상 ==========
번호 [0] -> 다음카페 수집 대상으로 지정되었습니다.
수집 기간 설정: 20240101 ~ 20260630 (총 30개 월별 구간 자동 생성)
키워드당 최대 수집 상한: 4500개 (자동: 30개월 x 150개) | 동시 실행 워커 수: 8개


### 1.2 제외할 카페 설정

In [34]:
# 제외할 카페 목록 (필수 실행. 목록은 없어도 됩니다.)
eliminate_cafe = [
    # 'https://cafe.naver.com/thswndhks123',
    # 'https://cafe.naver.com/koarane',
    # 'https://cafe.naver.com/sjs986'
]

print(f"제외할 카페: {len(eliminate_cafe)}개")

제외할 카페: 0개


### 1.3 키워드 조합 생성

In [35]:
# 키워드 조합 생성
keywords = []

for main in main_keywords:
    for inv in invest_key:
        keywords.append(main + ' +%2B' + inv + ' ' + eliminate_ads_keyword)

print("생성된 키워드:")
for i, kw in enumerate(keywords, 1):
    print(f"  {i}. {kw}")

생성된 키워드:
  1. 귀농 +%2B건강식 
  2. 귀농 +%2B아침 
  3. 귀농 +%2B운동 
  4. 귀농 +%2B산책 
  5. 귀농 +%2B부항 
  6. 귀농 +%2B함께운동 
  7. 귀농 +%2B제철음식 
  8. 귀농 +%2B식도염 
  9. 귀농 +%2B식사량 
  10. 귀농 +%2B뷰 
  11. 귀농 +%2B영양제 
  12. 귀농 +%2B아침루틴 
  13. 귀농 +%2B요리 
  14. 귀농 +%2B텃밭 
  15. 귀농 +%2B항산화 
  16. 귀농 +%2B위염 
  17. 귀농 +%2B역류성 
  18. 귀농 +%2B스트레스 
  19. 귀농 +%2B바이오리듬 
  20. 귀농 +%2B소화 
  21. 귀농 +%2B기온차 
  22. 귀농 +%2B일교차 
  23. 귀농 +%2B위건강 
  24. 귀농 +%2B해독주스 
  25. 귀농 +%2B식욕조절 
  26. 귀농 +%2B감기 
  27. 귀농 +%2B한약 
  28. 귀농 +%2B몸보신 
  29. 귀농 +%2B창문 
  30. 귀농 +%2B옆집 
  31. 귀농 +%2B소리 
  32. 귀농 +%2B냄새 
  33. 귀농 +%2B향기 
  34. 귀농 +%2B아침 
  35. 귀농 +%2B음악 
  36. 귀농 +%2B찬 음식 
  37. 귀농 +%2B계획 
  38. 귀농 +%2B환기 
  39. 귀농 +%2B공기 
  40. 귀농 +%2B빛 
  41. 귀농 +%2B소음 
  42. 귀농 +%2B온도 
  43. 귀농 +%2B등 
  44. 귀농 +%2B면역력 
  45. 귀농 +%2B시간 
  46. 귀농 +%2B보일러 
  47. 귀농 +%2B코골이 
  48. 귀농 +%2B암막 
  49. 귀농 +%2B이산화탄소 
  50. 귀농 +%2B피곤 
  51. 귀농 +%2B환절기 
  52. 귀농 +%2B으슬으슬 
  53. 귀농 +%2B노안 
  54. 귀농 +%2B자급자족 
  55. 귀농 +%2B혈당 
  56. 귀농 +%2B연당기 
  57. 귀농 +%2

### 1.4 네트워크 끊김 자동 대기
- 노트북을 닫거나 네트워크가 끊기면 수집을 잠시 정지하고, 다시 연결되면 자동으로 재개합니다.
- 아래 셀을 실행해 두기만 하면 됩니다. (헬퍼 함수 셀보다 먼저 실행)

In [36]:
# ==================== [네트워크 끊김 자동 대기] ====================
# 노트북 덮개를 닫거나 절전 모드에 들어가거나 와이파이가 끊기면 요청이 실패합니다.
# 이 셀을 실행해 두면 fetch_html이 네트워크 단절을 감지했을 때 모든 워커가 멈추고,
# 네트워크가 다시 연결되면 실패한 요청부터 자동으로 이어서 수집합니다. (실패로 기록되지 않습니다)
# 반드시 헬퍼 함수 셀(2번 항목)보다 먼저 실행하세요.

NETWORK_CHECK_HOST = ("search.daum.net", 443)  # 연결 확인용 주소
NETWORK_POLL_SEC = 5                           # 끊긴 동안 연결 확인 주기(초)

_network_ok = threading.Event()   # set: 정상 / clear: 끊김(모든 워커 대기)
_network_ok.set()
_network_lock = threading.Lock()  # 끊김 동안 연결 확인은 한 스레드만 수행

# 수집 중단(Interrupt) 신호: 세워지면 모든 워커가 다음 요청·대기 전에 빠져나옵니다
STOP = threading.Event()


class Stopped(BaseException):
    """수집 중단 요청으로 작업을 그만둔 경우(저장하지 않음, 재실행 시 이어서 수집).
    글 단위 오류 처리(except Exception)에 걸리지 않도록 BaseException을 상속합니다."""


def is_network_up(timeout=3):
    try:
        with socket.create_connection(NETWORK_CHECK_HOST, timeout=timeout):
            return True
    except OSError:
        return False


def is_network_error(exc):
    """HTTP 응답 오류가 아닌 연결 계열 오류(끊김, 타임아웃, DNS 실패 등)인지 판단"""
    if isinstance(exc, urllib.error.HTTPError):
        return False
    return isinstance(exc, (urllib.error.URLError, socket.timeout, TimeoutError, ConnectionError, OSError))


def wait_for_network():
    """
    네트워크가 실제로 끊긴 상태이면 복구될 때까지 모든 워커를 멈추고 True를 반환합니다.
    네트워크가 정상이면(서버 쪽 일시 오류 등) 기다리지 않고 False를 반환합니다.
    """
    if _network_ok.is_set() and is_network_up():
        return False
    with _network_lock:
        if _network_ok.is_set():
            _network_ok.clear()
            started = time.time()
            print(f"[{time.strftime('%H:%M:%S')}] 네트워크 연결 끊김 감지 - 수집을 일시 정지합니다. (연결되면 자동으로 재개)")
            while not STOP.is_set() and not is_network_up():
                STOP.wait(NETWORK_POLL_SEC)
            _network_ok.set()
            print(f"[{time.strftime('%H:%M:%S')}] 네트워크 복구 - 수집을 재개합니다. (정지 {int(time.time() - started)}초)")
    _network_ok.wait()
    return True


print(f"네트워크 끊김 자동 대기 설정 완료 (확인 주소: {NETWORK_CHECK_HOST[0]}, 확인 주기: {NETWORK_POLL_SEC}초)")


네트워크 끊김 자동 대기 설정 완료 (확인 주소: search.daum.net, 확인 주기: 5초)


## 2. 헬퍼 함수 정의

In [37]:
DEFAULT_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}

# ---------- 글 캐시 (SQLite, 디스크) ----------
# 받은 글(posts)과 삭제된 글(dead, 404/410)을 노트북 폴더의 SQLite 파일에 저장합니다.
# 메모리에 올리지 않아 data_csv가 커져도 메모리가 늘지 않고, 커널을 재시작해도 그대로 이어서 씁니다.
# 지워도 data_csv의 CSV로 다시 만들어집니다(지울 때는 -wal, -shm 파일도 함께). iCloud 같은 동기화 폴더가 아닌 로컬 폴더에 두세요.
import sqlite3

CACHE_DB = "./post_cache.sqlite"
POST_FIELDS = ["date", "title", "contents", "comments", "author_nickname", "comment_nicknames"]
_db = sqlite3.connect(CACHE_DB, check_same_thread=False, isolation_level=None)
_db.execute("PRAGMA journal_mode=WAL")
_db.execute("PRAGMA synchronous=NORMAL")
_db.execute("CREATE TABLE IF NOT EXISTS posts (url TEXT PRIMARY KEY, data TEXT NOT NULL)")   # 정규화 URL -> 글 정보
_db.execute("CREATE TABLE IF NOT EXISTS dead (url TEXT PRIMARY KEY)")                       # 404/410을 받은 URL
_db.execute("CREATE TABLE IF NOT EXISTS imported (name TEXT PRIMARY KEY, size INTEGER)")    # 캐시에 반영 완료한 CSV
_db_lock = threading.Lock()


def cache_get(norm_url):
    with _db_lock:
        row = _db.execute("SELECT data FROM posts WHERE url=?", (norm_url,)).fetchone()
    return json.loads(row[0]) if row else None


def cache_put(norm_url, data):
    with _db_lock:
        _db.execute("INSERT OR REPLACE INTO posts VALUES (?, ?)",
                    (norm_url, json.dumps({k: data[k] for k in POST_FIELDS}, ensure_ascii=False, default=str)))


def is_dead(url):
    with _db_lock:
        return _db.execute("SELECT 1 FROM dead WHERE url=?", (url,)).fetchone() is not None


def mark_dead(url):
    with _db_lock:
        _db.execute("INSERT OR IGNORE INTO dead VALUES (?)", (url,))


def is_imported(path):
    with _db_lock:
        row = _db.execute("SELECT size FROM imported WHERE name=?", (os.path.basename(path),)).fetchone()
    return row is not None and row[0] == os.path.getsize(path)


def mark_imported(path):
    with _db_lock:
        _db.execute("INSERT OR REPLACE INTO imported VALUES (?, ?)", (os.path.basename(path), os.path.getsize(path)))


class FetchError(Exception):
    """재시도 후에도 요청이 실패한 경우(차단, 서버 오류 등)"""


def fetch_html(url, referer=None, timeout=10, retries=3, strict=False):
    """
    브라우저 창을 띄우지 않고 URL에서 HTML 원문을 직접 받아오는 함수

    - 네트워크 단절(노트북 닫힘 등)은 연결이 복구될 때까지 대기한 뒤 같은 요청을 다시 보냅니다.
      (재시도 횟수를 소모하지 않음, 1.4 셀 필요)
    - 429/403/5xx 등 서버 오류는 지수 백오프로 retries회까지 재시도합니다. 404/410은 재시도하지 않습니다.
    - 재시도 후에도 실패하면 strict=True일 때 FetchError를 발생시키고, 아니면 None을 반환합니다.
      (검색 페이지는 strict=True로 호출해, 실패를 '결과 없음'으로 오인하지 않게 합니다)
    - 수집 중단 요청(STOP)이 있으면 Stopped를 발생시킵니다.
    """
    last_err = None
    attempt = 0
    while attempt < retries:
        _network_ok.wait()
        if STOP.is_set():
            raise Stopped()
        try:
            cj = http.cookiejar.CookieJar()
            opener = urllib.request.build_opener(urllib.request.HTTPCookieProcessor(cj))
            headers = dict(DEFAULT_HEADERS)
            if referer:
                headers["Referer"] = referer
            req = urllib.request.Request(url, headers=headers)
            with opener.open(req, timeout=timeout) as resp:
                raw_bytes = resp.read()
                charset = resp.headers.get_content_charset() or "utf-8"
                return raw_bytes.decode(charset, errors="ignore")
        except urllib.error.HTTPError as e:
            if e.code in (404, 410):
                mark_dead(url)
                return None
            last_err = e
        except Exception as e:
            last_err = e
            if is_network_error(e) and wait_for_network():
                continue
        attempt += 1
        if attempt < retries:
            STOP.wait(2 ** (attempt - 1) + random.uniform(0, 1))
    if strict:
        raise FetchError(f"{url} ({last_err})")
    return None


NO_RESULT_TEXT = "검색결과가 없습니다"


def fetch_search_page(url):
    """
    Daum 검색 페이지 요청. 실패하면 FetchError.
    검색어 보정 등으로 내려오는 JS 이동 스텁(location.replace)은 따라가 실제 결과 페이지를 받아옵니다.
    """
    html = fetch_html(url, strict=True)
    for _ in range(2):
        m = re.search(r'location\.replace\("([^"]+)"\)', html) if html and len(html) < 2000 else None
        if not m:
            break
        html = fetch_html(urllib.parse.urljoin(url, m.group(1)), referer=url, strict=True)
    if html is None:
        raise FetchError(f"{url} (응답 없음)")
    return html


def is_confirmed_empty(search_html):
    """링크가 0개인 검색 페이지가 '정말 결과 없음' 안내 페이지인지 확인 (차단/오류 페이지와 구분)"""
    return bool(search_html) and NO_RESULT_TEXT in search_html


def resolve_post_html(url, html_source, target_id=0):
    """
    다음카페(target_id == 0)인 경우 상위 페이지 내 down 프레임의 src 주소를 찾아
    실제 게시글 본문 HTML을 반환하고, 그 외 대상은 원본 HTML을 그대로 반환하는 함수
    """
    if not html_source:
        return None
    if target_id != 0:
        return html_source

    try:
        soup = bs(html_source, "lxml")
        down_frame = soup.select_one("iframe#down, iframe[name='down']")
        if down_frame and down_frame.get("src"):
            iframe_url = urllib.parse.urljoin("https://cafe.daum.net", down_frame["src"])
            return fetch_html(iframe_url, referer=url)
        return html_source
    except Exception as e:
        pass
        return None


def extract_post_data(html_source, target_id=0):
    """
    게시물 HTML에서 데이터 추출 함수

    Args:
        html_source: 게시글 HTML 문자열
        target_id: 수집 대상 번호 (0: 다음카페, 1: 티스토리, 2: 브런치, 3: 웹문서, 4: 다음뉴스)

    Returns:
        dict or None: 추출된 데이터 또는 실패시 None
    """
    if not html_source:
        return None

    try:
        soup = bs(html_source, "lxml")

        # 0: 다음카페
        if target_id == 0:
            title_elem = soup.select_one("span.article_title, strong.tit_info, h3.tit_subject")
            if not title_elem:
                return None
            title = title_elem.get_text(strip=True)

            contents_elems = soup.select("div#user_contents, div.article_view")
            contents = " ".join([i.get_text(" ", strip=True) for i in contents_elems if i.get_text(strip=True)])
            if not contents:
                xmp_elem = soup.select_one("xmp#template_xmp")
                if xmp_elem and xmp_elem.string:
                    contents = bs(xmp_elem.string, "lxml").get_text(" ", strip=True)
            contents = contents.replace("\u200b", " ")
            contents = re.sub(r"\s+", " ", contents).strip()

            date = ""
            for item in soup.select("div.cover_info span.txt_item, span.num_date, span.date"):
                txt = item.get_text(strip=True)
                if re.search(r"\d{2,4}\.\d{1,2}\.\d{1,2}", txt):
                    date = txt
                    break

            comments = [i.get_text(" ", strip=True) for i in soup.select("span.original_comment")]

            try:
                author_elem = soup.select_one("div.cover_info a.link_item, span.txt_writer")
                author_nickname = author_elem.get_text(strip=True) if author_elem else "닉네임 없음"
            except:
                author_nickname = "닉네임 없음"

            try:
                comment_nick_elems = soup.select("div.comment_view a.txt_name")
                if comment_nick_elems:
                    comment_nicknames = " | ".join([n.get_text(strip=True) for n in comment_nick_elems])
                else:
                    comment_nicknames = "댓글 닉네임 없음"
            except:
                comment_nicknames = "댓글 닉네임 없음"

        # 1: 다음블로그_티스토리
        elif target_id == 1:
            og_title = soup.select_one("meta[property='og:title']")
            title_elem = soup.select_one("div.hgroup h1, h2.tit_post, .titleWrap h2, h1")
            title = og_title["content"].strip() if (og_title and og_title.get("content")) else (title_elem.get_text(strip=True) if title_elem else "")
            if not title:
                return None

            contents_elems = soup.select("div.contents_style, div.tt_article_useless_p_margin, div.article-view, div.entry-content, article")
            contents = " ".join([i.get_text(" ", strip=True) for i in contents_elems])
            contents = contents.replace("\u200b", " ")
            contents = re.sub(r"\s+", " ", contents).strip()

            pub_meta = soup.select_one("meta[property='article:published_time'], meta[property='og:regDate']")
            date_elem = soup.select_one("span.date, time")
            date = pub_meta["content"].strip() if (pub_meta and pub_meta.get("content")) else (date_elem.get_text(strip=True) if date_elem else "")

            comments = [i.get_text(" ", strip=True) for i in soup.select("div.rp_general p, div.comment-content, p.txt_reply, div.tt-box-content .tt-desc")]

            author_meta = soup.select_one("meta[name='author'], meta[property='og:site_name']")
            author_nickname = author_meta["content"].strip() if (author_meta and author_meta.get("content")) else "닉네임 없음"

            comment_nick_elems = soup.select("span.name, .tt-box-meta .tt-link-user, .rp_general .name")
            comment_nicknames = " | ".join([n.get_text(strip=True) for n in comment_nick_elems]) if comment_nick_elems else "댓글 닉네임 없음"

        # 2: 다음블로그_브런치
        elif target_id == 2:
            title_elem = soup.select_one("h1.cover_title")
            og_title = soup.select_one("meta[property='og:title']")
            title = title_elem.get_text(strip=True) if title_elem else (og_title["content"].strip() if (og_title and og_title.get("content")) else "")
            if not title:
                return None

            contents_elems = soup.select("div.wrap_body, div.wrap_body_frame")
            contents = " ".join([i.get_text(" ", strip=True) for i in contents_elems])
            contents = contents.replace("\u200b", " ")
            contents = re.sub(r"\s+", " ", contents).strip()

            pub_meta = soup.select_one("meta[property='article:published_time'], meta[property='og:regDate']")
            date_elem = soup.select_one("span.f_nb, span.date")
            date = pub_meta["content"].strip() if (pub_meta and pub_meta.get("content")) else (date_elem.get_text(strip=True) if date_elem else "")

            comments = [i.get_text(" ", strip=True) for i in soup.select("div.comment_content p.desc_comment, .list_comment .desc_comment")]

            author_meta = soup.select_one("meta[name='author']")
            author_elem = soup.select_one("span.by_user a, strong.author_name")
            author_nickname = author_meta["content"].strip() if (author_meta and author_meta.get("content")) else (author_elem.get_text(strip=True) if author_elem else "닉네임 없음")

            comment_nick_elems = soup.select(".list_comment .link_nick, .comment_head .name")
            comment_nicknames = " | ".join([n.get_text(strip=True) for n in comment_nick_elems]) if comment_nick_elems else "댓글 닉네임 없음"

        # 3: 다음웹문서
        elif target_id == 3:
            og_title = soup.select_one("meta[property='og:title']")
            title_elem = soup.select_one("h1, h2, title")
            title = og_title["content"].strip() if (og_title and og_title.get("content")) else (title_elem.get_text(strip=True) if title_elem else "")
            if not title:
                return None

            contents_elems = soup.select("article, main, div.write_div, div.rd_body, div.post_content, div.view_content, div#body")
            if not contents_elems:
                contents_elems = soup.select("body p")
            contents = " ".join([i.get_text(" ", strip=True) for i in contents_elems])
            contents = contents.replace("\u200b", " ")
            contents = re.sub(r"\s+", " ", contents).strip()

            pub_meta = soup.select_one("meta[property='article:published_time']")
            date_elem = soup.select_one("time, span.date, span.gall_date")
            date = pub_meta["content"].strip() if (pub_meta and pub_meta.get("content")) else (date_elem.get_text(strip=True) if date_elem else "")

            comments = [i.get_text(" ", strip=True) for i in soup.select(".ub-content .usertxt, .comment_content, .fdb_lst_ul .xe_content")]

            author_meta = soup.select_one("meta[name='author']")
            author_elem = soup.select_one(".gall_writer .nickname, .member_plate, .author")
            author_nickname = author_meta["content"].strip() if (author_meta and author_meta.get("content")) else (author_elem.get_text(strip=True) if author_elem else "닉네임 없음")

            comment_nick_elems = soup.select(".ub-writer .nickname, .comment_nick, .meta .author")
            comment_nicknames = " | ".join([n.get_text(strip=True) for n in comment_nick_elems]) if comment_nick_elems else "댓글 닉네임 없음"

        # 4: 다음뉴스
        elif target_id == 4:
            title_elem = soup.select_one("h3.tit_view")
            og_title = soup.select_one("meta[property='og:title']")
            title = title_elem.get_text(strip=True) if title_elem else (og_title["content"].strip() if (og_title and og_title.get("content")) else "")
            if not title:
                return None

            contents_elems = soup.select("div.article_view section, div.article_view")
            contents = " ".join([i.get_text(" ", strip=True) for i in contents_elems[:1]])
            contents = contents.replace("\u200b", " ")
            contents = re.sub(r"\s+", " ", contents).strip()

            date_elem = soup.select_one("span.num_date")
            reg_meta = soup.select_one("meta[property='og:regDate']")
            date = date_elem.get_text(strip=True) if date_elem else (reg_meta["content"].strip() if (reg_meta and reg_meta.get("content")) else "")

            comments = [i.get_text(" ", strip=True) for i in soup.select("ul.list_comment p.desc_txt")]

            author_elem = soup.select_one("span.txt_info")
            author_nickname = author_elem.get_text(strip=True) if author_elem else "닉네임 없음"

            comment_nick_elems = soup.select("ul.list_comment a.link_nick, ul.list_comment span.txt_name")
            comment_nicknames = " | ".join([n.get_text(strip=True) for n in comment_nick_elems]) if comment_nick_elems else "댓글 닉네임 없음"

        else:
            return None

        error_keywords = ["사이트에 연결할 수 없음", "연결이 비공개로 설정되어 있지 않습니다", "ERR_"]
        if any(err in title for err in error_keywords):
            return None

        return {
            "title": title,
            "contents": contents,
            "date": date,
            "comments": comments,
            "author_nickname": author_nickname,
            "comment_nicknames": comment_nicknames
        }
    except Exception as e:
        pass
        return None


# ---------- 브런치 댓글 수집 ----------
# 댓글을 못 받거나 개수가 어긋나도 수집을 멈추지 않습니다. 해당 글은 경고로 기록하고(수집_경고.txt) 계속 진행합니다.
# (네트워크 끊김은 fetch_html이 복구될 때까지 대기하므로 여기서 다루지 않습니다)
class BrunchCommentError(Exception):
    """브런치 댓글을 수집하지 못한 사유 (경고로 기록하고 수집은 계속)"""


BRUNCH_COMMENT_URL = "https://api.brunch.co.kr/v2/@@{uid}/{no}/comments"
BRUNCH_REPLY_URL = "https://api.brunch.co.kr/v1/@@{uid}/{no}/comment/{cno}/children"
COMMENT_FAIL_MARK = "댓글 수집 실패"  # 댓글 수집에 실패한 글의 comment_nicknames 값 (나중에 골라서 재수집 가능)


def _brunch_api_data(url, referer):
    """브런치 댓글 API 호출 후 data 딕셔너리 반환. 실패/예상과 다른 응답이면 BrunchCommentError"""
    try:
        body = fetch_html(url, referer=referer, strict=True)
    except FetchError as e:
        raise BrunchCommentError(f"댓글 API 요청이 재시도 후에도 실패: {e}")
    try:
        payload = json.loads(body)
        data = payload["data"]
        if payload.get("code") != 200 or not isinstance(data["list"], list):
            raise ValueError("unexpected payload")
    except Exception:
        raise BrunchCommentError(f"댓글 API 응답이 예상과 다름(구조 변경/차단 의심): {url}")
    time.sleep(random.uniform(0.25, 0.75))
    return data


def _collect_brunch_comments(html_source, url):
    soup = bs(html_source, "lxml")
    store = soup.select_one("script#COMMENT_STORE")
    store_text = (store.get_text() if store else "").strip()
    if not store_text.isdigit():
        raise BrunchCommentError("댓글 수(COMMENT_STORE)를 찾지 못함(페이지 구조 변경 의심)")
    expected = int(store_text)
    if expected == 0:
        return [], "댓글 닉네임 없음", None

    m_uid = re.search(r'article:author"\s+content="https?://brunch\.co\.kr/@@(\w+)"', html_source)
    m_no = re.search(r"brunch\.co\.kr/@[^/]+/(\d+)", url)
    if not (m_uid and m_no):
        raise BrunchCommentError("작가 ID/글 번호를 찾지 못함")
    uid, article_no = m_uid.group(1), m_no.group(1)

    # 최상위 댓글: 응답의 moreList가 True면 가장 오래된 댓글 번호를 commentNo로 넘겨 이전 댓글을 이어서 받습니다
    tops = {}
    cursor = None
    while True:
        api = BRUNCH_COMMENT_URL.format(uid=uid, no=article_no) + (f"?commentNo={cursor}" if cursor else "")
        data = _brunch_api_data(api, url)
        for c in data["list"]:
            tops[c["no"]] = c
        if not data.get("moreList"):
            break
        new_cursor = min(c["no"] for c in data["list"]) if data["list"] else None
        if new_cursor is None or new_cursor == cursor:
            raise BrunchCommentError("댓글 페이지 이동이 진행되지 않음")
        cursor = new_cursor

    # 답글: children.moreList가 True면 가장 오래된 답글 번호를 childNo로 넘겨 이어서 받습니다
    ordered = []
    for top in sorted(tops.values(), key=lambda c: c["createTime"]):
        children = {ch["no"]: ch for ch in top["children"]["list"]}
        more = top["children"].get("moreList")
        child_cursor = None
        while more:
            new_cursor = min(children) if children else None
            if new_cursor is None or new_cursor == child_cursor:
                raise BrunchCommentError("답글 페이지 이동이 진행되지 않음")
            child_cursor = new_cursor
            data = _brunch_api_data(BRUNCH_REPLY_URL.format(uid=uid, no=article_no, cno=top["no"]) + f"?childNo={child_cursor}", url)
            for ch in data["list"]:
                children[ch["no"]] = ch
            more = data.get("moreList")
        ordered.append(top)
        ordered.extend(sorted(children.values(), key=lambda c: c["createTime"]))

    # 삭제된 댓글은 본문이 "삭제된 댓글입니다."라서 수집하지 않고, 정상 댓글(NORMAL)만 저장합니다.
    # 화면에 표시되는 댓글 수는 삭제 유형에 따라 삭제 댓글을 세기도 하고 안 세기도 하므로(예: DELETED는 제외,
    # DELETED_DONATION은 포함), 세 가지 기준 중 하나라도 표시 수와 맞으면 정상, 모두 안 맞으면 경고만 기록합니다.
    visible = [c for c in ordered if c.get("deleteStatus") == "NORMAL"]
    n_not_deleted = sum(1 for c in ordered if c.get("deleteStatus") != "DELETED")
    warning = None
    if expected not in (len(visible), n_not_deleted, len(ordered)):
        statuses = collections.Counter(c.get("deleteStatus") for c in ordered)
        warning = f"댓글 수 불일치(표시 {expected}개 / 정상 {len(visible)}개, 삭제 제외 {n_not_deleted}개, 전체 {len(ordered)}개, 상태 {dict(statuses)})"
    messages = [c["message"] for c in visible]
    nicknames = " | ".join(c["commentUserName"] for c in visible) if visible else "댓글 닉네임 없음"
    return messages, nicknames, warning


def fetch_brunch_comments(html_source, url):
    """
    브런치 글의 댓글 원문 전체(답글 포함) 수집.
    Returns: (댓글 본문 리스트, " | "로 연결한 작성자 닉네임, 경고 문구 또는 None)
    어떤 경우에도 예외를 내지 않아 수집이 멈추지 않습니다. 수집에 실패하면 ([], "댓글 수집 실패", 사유)를 반환합니다.
    """
    try:
        return _collect_brunch_comments(html_source, url)
    except BrunchCommentError as e:
        return [], COMMENT_FAIL_MARK, str(e)
    except Exception as e:
        return [], COMMENT_FAIL_MARK, f"예상치 못한 오류 {type(e).__name__}: {e}"


# ---------- 티스토리 댓글 수집 ----------
# 티스토리 댓글도 HTML에는 없고 블로그 주소 아래의 /m/api/{글번호}/comment API로 받아옵니다. (커스텀 도메인 블로그도 동일)
# 비밀댓글은 원문이 없어("비밀댓글입니다.") 수집하지 않습니다. 문제가 생겨도 멈추지 않고 경고로 기록합니다.
class TistoryCommentError(Exception):
    """티스토리 댓글을 수집하지 못한 사유 (경고로 기록하고 수집은 계속)"""


def _tistory_api_data(url, referer):
    try:
        body = fetch_html(url, referer=referer, strict=True)
    except FetchError as e:
        raise TistoryCommentError(f"댓글 API 요청이 재시도 후에도 실패: {e}")
    try:
        data = json.loads(body)["data"]
        if not isinstance(data["items"], list):
            raise ValueError("unexpected payload")
    except Exception:
        raise TistoryCommentError(f"댓글 API 응답이 예상과 다름(구조 변경/차단/삭제된 글 의심): {url}")
    time.sleep(random.uniform(0.25, 0.75))
    return data


def _tistory_entry_id(url, html_source):
    path = urllib.parse.urlparse(url).path.strip("/")
    m = re.fullmatch(r"(?:entry/)?(\d+)", path)
    if m:
        return m.group(1)
    if html_source:  # 글 제목이 주소인 글: HTML 안의 글 번호 사용
        m = re.search(r'"?entryId"?\s*[:=]\s*"?(\d+)', html_source)
        if m:
            return m.group(1)
    return None


def _collect_tistory_comments(html_source, url):
    p = urllib.parse.urlparse(url)
    base = f"{p.scheme}://{p.netloc}"
    entry_id = _tistory_entry_id(url, html_source)
    if not entry_id:
        raise TistoryCommentError("글 번호를 찾지 못함")

    # 최상위 댓글: isLast가 False면 nextId를 startId로 넘겨 이전 댓글을 이어서 받습니다 (한 번에 20개)
    tops = {}
    start_id = None
    total_items = None
    while True:
        api = f"{base}/m/api/{entry_id}/comment?reverse=true" + (f"&startId={start_id}" if start_id else "")
        data = _tistory_api_data(api, url)
        if total_items is None:
            total_items = data.get("totalItems")
        for it in data["items"]:
            tops[it["id"]] = it
        if data.get("isLast") or not data["items"]:
            break
        new_start = data.get("nextId")
        if not new_start or new_start == start_id:
            raise TistoryCommentError("댓글 페이지 이동이 진행되지 않음")
        start_id = new_start

    ordered = []
    for top in sorted(tops.values(), key=lambda c: c["id"]):
        ordered.append(top)
        ordered.extend(sorted(top.get("children") or [], key=lambda c: c["id"]))

    warning = None
    if total_items is not None and len(tops) != total_items:
        warning = f"댓글 수 불일치(표시 {total_items}개 / 수집한 최상위 댓글 {len(tops)}개)"
    visible = [c for c in ordered if not c.get("isSecret") and (c.get("content") or "").strip()]
    messages = [c["content"] for c in visible]
    nicknames = " | ".join(((c.get("writer") or {}).get("name") or "") for c in visible) if visible else "댓글 닉네임 없음"
    return messages, nicknames, warning


def fetch_tistory_comments(html_source, url):
    """
    티스토리 글의 댓글 원문(답글 포함) 수집. Returns: (댓글 본문 리스트, " | "로 연결한 닉네임, 경고 문구 또는 None)
    어떤 경우에도 예외를 내지 않아 수집이 멈추지 않습니다. 실패하면 ([], "댓글 수집 실패", 사유)를 반환합니다.
    """
    try:
        return _collect_tistory_comments(html_source, url)
    except TistoryCommentError as e:
        return [], COMMENT_FAIL_MARK, str(e)
    except Exception as e:
        return [], COMMENT_FAIL_MARK, f"예상치 못한 오류 {type(e).__name__}: {e}"


# ---------- 본문이 너무 긴 글 블랙리스트 ----------
# VOC로 보기 어려운 긴 글(MAX_CONTENT_CHARS 초과)은 저장하지 않고 URL을 블랙리스트에 올립니다.
# 이후 다른 키워드에서 같은 글이 나오면 다시 받지 않고 건너뜁니다. 파일로도 남겨 재시작해도 유지됩니다.
BLACKLIST_FILE = "./수집_블랙리스트.txt"
BLACKLIST_LOCK = threading.Lock()
if "BLACKLIST" not in globals():
    BLACKLIST = set()


def load_blacklist():
    if os.path.exists(BLACKLIST_FILE):
        with open(BLACKLIST_FILE, encoding="utf-8") as f:
            for line in f:
                u = line.split("\t", 1)[0].strip()
                if u:
                    BLACKLIST.add(u)
    return len(BLACKLIST)


def add_blacklist(norm_url, length):
    with BLACKLIST_LOCK:
        if norm_url in BLACKLIST:
            return
        BLACKLIST.add(norm_url)
        with open(BLACKLIST_FILE, "a", encoding="utf-8") as f:
            f.write(f"{norm_url}\t{length}\n")


print("헬퍼 함수 정의")

헬퍼 함수 정의


## 3. 병렬배치 설정
- 설정값에 따라 속도를 배로 올릴 수 있습니다.
- 컴퓨터가 뜨거워질 수 있습니다.

In [38]:
# 병렬워커 수 확인 (1.1 상단 통합 설정 셀에서 통합 관리)
if "MAX_WORKERS" not in globals():
    MAX_WORKERS = 8

print(f"HTML 직접 수집 다중배치 설정 완료 (동시 실행 워커 수: {MAX_WORKERS}개)")


HTML 직접 수집 다중배치 설정 완료 (동시 실행 워커 수: 8개)


## 4. 제외 URL·이어하기 설정
### 4-1. 제외 URL 설정 (overlap_data)
- 이미 수집한 CSV 파일 속 URL을 추출하여 수집대상에서 제외합니다.
- 기존에 수집 완료된 csv파일은 overlap_data 폴더에 두시면 됩니다.

In [39]:
# overlap_data 폴더 내 CSV 파일 로드 → 제외할 URL 추출
import urllib.parse

def normalize_url(url):
    """URL에서 쿼리스트링 제거 후 기본 경로만 추출"""
    if pd.isna(url):
        return ""
    parsed = urllib.parse.urlparse(str(url))
    # scheme + netloc + path 만 남김 (? 이후 파라미터 제거)
    return f"{parsed.scheme}://{parsed.netloc}{parsed.path}"

existing_urls = set()
overlap_csv_files = glob.glob(os.path.join(OVERLAP_PATH, "*.csv"))
total_loaded_rows = 0

for csv_file in overlap_csv_files:
    try:
        df_existing = pd.read_csv(csv_file, encoding="utf-8-sig")
        if "url" in df_existing.columns:
            total_loaded_rows += len(df_existing)
            existing_urls.update(df_existing["url"].dropna().apply(normalize_url))
            print(f"로드 완료: {os.path.basename(csv_file)} ({len(df_existing)}행)")
        else:
            print(f"[경고] url 컬럼 없음 (스킵): {os.path.basename(csv_file)}")
    except Exception as e:
        print(f"[오류] 파일 읽기 오류 ({os.path.basename(csv_file)}): {e}")

existing_urls.discard("")
print(f"overlap_data 내 CSV 파일 수: {len(overlap_csv_files)}개 (총 {total_loaded_rows}행)")
print(f"제외할 고유 URL 수: {len(existing_urls)}개")

overlap_data 내 CSV 파일 수: 0개 (총 0행)
제외할 고유 URL 수: 0개


### 4-2. 이어하기·캐시 설정
- `data_csv/`에 이미 저장된 키워드는 건너뜁니다. 0개 파일은 '검증된 0건' 기록(`_verified_zero.tsv`)이 있을 때만 완료로 봅니다.
- 받은 글을 SQLite 캐시(`post_cache.sqlite`)에 넣어, 다른 키워드에서 같은 글이 나오면 다시 요청하지 않고 복사합니다. 처음 한 번은 기존 CSV를 캐시에 옮기느라 오래 걸릴 수 있습니다.
- 수집을 중단했다가 다시 시작할 때는 이 셀부터 실행하세요.

In [40]:
# 데이터프레임 초기화 및 이어하기(기존 완료 파일 스킵) 설정
RESUME_FROM_SAVED = True  # True: data_csv에 이미 저장된 키워드 파일은 건너뛰고 미수집 키워드만 이어서 수집 (0개 파일은 미완료로 보고 재수집)

dataframe = pd.DataFrame(columns=["date", "keyword", "title", "contents", "comments", "site", "url", "author_nickname", "comment_nicknames"])
index = 0

completed_task_keys = set()
if RESUME_FROM_SAVED:
    saved_pattern = re.compile(r"^다음포탈_(?P<site>다음카페|다음블로그_티스토리|다음블로그_브런치|다음웹문서|다음뉴스)_(?P<safe_kw>.+)_[1-9]\d*개\.csv$")
    for fpath in glob.glob(os.path.join(SAVE_PATH, "다음포탈_*.csv")):
        fname = os.path.basename(fpath)
        m = saved_pattern.match(fname)
        if m:
            completed_task_keys.add((m.group("site"), m.group("safe_kw")))

# 검증된 0건: 모든 검색 요청이 성공하고 매번 '검색결과가 없습니다' 안내 페이지를 확인한 키워드 (수집 시 자동 기록)
VERIFIED_ZERO_FILE = os.path.join(SAVE_PATH, "_verified_zero.tsv")
verified_zero_keys = set()
if os.path.exists(VERIFIED_ZERO_FILE):
    with open(VERIFIED_ZERO_FILE, encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 2:
                verified_zero_keys.add((parts[0], parts[1]))
if RESUME_FROM_SAVED:
    completed_task_keys |= verified_zero_keys

# 검증 기록이 없는 0개 파일(이전 실행분): 네트워크 오류로 0개가 됐는지 알 수 없으므로 재수집 대상
unverified_zero = []
zero_pattern = re.compile(r"^다음포탈_(?P<site>다음카페|다음블로그_티스토리|다음블로그_브런치|다음웹문서|다음뉴스)_(?P<safe_kw>.+)_0개\.csv$")
for fpath in glob.glob(os.path.join(SAVE_PATH, "다음포탈_*_0개.csv")):
    m = zero_pattern.match(os.path.basename(fpath))
    if m and (m.group("site"), m.group("safe_kw")) not in verified_zero_keys:
        unverified_zero.append(os.path.basename(fpath))

total_planned = 0
already_done = 0
for t_id in SELECTED_TARGETS:
    if t_id in DAUM_TARGETS:
        t_name = DAUM_TARGETS[t_id]["name"]
        for kw in keywords:
            total_planned += 1
            safe_kw = kw.replace("+%2B", "_").replace(" ", "").replace("/", "_")
            if (t_name, safe_kw) in completed_task_keys:
                already_done += 1

print("데이터프레임 초기화 및 이어하기 점검 완료!")
print(f" - 이어하기 모드(RESUME_FROM_SAVED): {RESUME_FROM_SAVED}")
print(f" - 0건 검증 기록: 검증된 0건 {len(verified_zero_keys)}개 | 검증 안 된 0개 파일 {len(unverified_zero)}개 (재수집 대상)")
print(f" - 전체 계획 작업: {total_planned}개 | 기존 완료(스킵 대상): {already_done}개 | 남은 수집 대상: {total_planned - already_done}개")

# URL별 수집 결과 캐시(SQLite, 2번 셀): 다른 키워드에서 같은 글이 또 검색되면 다시 요청하지 않고 기존 정보를 복사합니다.
# data_csv의 CSV 중 아직 캐시에 반영하지 않은 파일(새 파일, 크기가 바뀐 파일)만 읽어 넣습니다. 처음 한 번은 오래 걸릴 수 있습니다.
pending_import = []
for t_id in SELECTED_TARGETS:
    if t_id not in DAUM_TARGETS:
        continue
    t_name = DAUM_TARGETS[t_id]["name"]
    for fpath in glob.glob(os.path.join(SAVE_PATH, f"다음포탈_{t_name}_*.csv")):
        if not fpath.endswith("_0개.csv") and not is_imported(fpath):
            pending_import.append((fpath, t_id))

skipped_no_comment_files = 0
if pending_import:
    print(f" - 캐시 반영: CSV {len(pending_import)}개를 SQLite 캐시에 넣는 중... (끝날 때까지 다음 셀을 실행하지 마세요)", flush=True)
    t_import = time.time()
for n, (fpath, t_id) in enumerate(pending_import, 1):
    try:
        df_saved = pd.read_csv(fpath, encoding="utf-8-sig").fillna("")
    except Exception as e:
        print(f"[경고] 캐시 로드 실패 ({os.path.basename(fpath)}): {e}")
        continue
    # 브런치(2)/티스토리(1): 댓글 수집 기능 이전에 받은 CSV는 댓글이 비어 있어 복사하면 댓글이 누락되므로,
    # 댓글이 한 건이라도 들어 있는 파일(=댓글 수집 이후 파일)만 캐시에 넣습니다.
    if t_id in (1, 2) and not (df_saved["comments"].astype(str) != "[]").any():
        skipped_no_comment_files += 1
    else:
        batch = [(normalize_url(rec["url"]), json.dumps({k: rec[k] for k in POST_FIELDS}, ensure_ascii=False, default=str))
                 for rec in df_saved.to_dict("records")]
        with _db_lock:
            _db.execute("BEGIN")
            _db.executemany("INSERT OR IGNORE INTO posts VALUES (?, ?)", batch)
            _db.execute("COMMIT")
    mark_imported(fpath)
    if n % 200 == 0 or n == len(pending_import):
        print(f"   {n}/{len(pending_import)}개 반영 (경과 {int(time.time() - t_import)}초)", flush=True)
with _db_lock:
    n_cached = _db.execute("SELECT COUNT(*) FROM posts").fetchone()[0]
    n_dead = _db.execute("SELECT COUNT(*) FROM dead").fetchone()[0]
print(f" - URL 캐시({CACHE_DB}): 고유 게시글 {n_cached}개, 삭제 확인 글 {n_dead}개" + (f" (이번에 반영한 파일 중 댓글 정보 없는 브런치/티스토리 파일 {skipped_no_comment_files}개는 캐시 제외)" if skipped_no_comment_files else ""))
print(f" - 블랙리스트(본문 {MAX_CONTENT_CHARS:,}자 초과 글) 로드: {load_blacklist()}개" if MAX_CONTENT_CHARS else " - 본문 길이 제한 없음(블랙리스트 사용 안 함)")


데이터프레임 초기화 및 이어하기 점검 완료!
 - 이어하기 모드(RESUME_FROM_SAVED): True
 - 0건 검증 기록: 검증된 0건 68개 | 검증 안 된 0개 파일 0개 (재수집 대상)
 - 전체 계획 작업: 1680개 | 기존 완료(스킵 대상): 0개 | 남은 수집 대상: 1680개
 - URL 캐시(./post_cache.sqlite): 고유 게시글 52698개, 삭제 확인 글 0개
 - 본문 길이 제한 없음(블랙리스트 사용 안 함)


## 5. 크롤링 실행

- 한번에 검색 가능한 페이지 수에 한계가 있어 상단 설정 셀에서 지정한 기간(`START_DATE` ~ `END_DATE`)을 월별로 분할하여 관련도순으로 수집합니다.
- 최대 target은 월별 데이터를 다 합쳐서 키워드별로 `TARGET_COUNT`입니다. (기본값은 기간의 월 수 x 150으로 자동 계산되며, 상단 설정 셀에서 숫자로 바꿀 수 있습니다)
- 4-2 셀의 이어하기 설정(`RESUME_FROM_SAVED = True`)이 켜져 있으면 이미 `data_csv`에 저장된 키워드는 자동으로 건너뛰고 남은 키워드부터 수집합니다.

In [41]:
if "YEAR_RANGES" not in globals():
    YEAR_RANGES = generate_monthly_ranges(START_DATE, END_DATE)
if "TARGET_COUNT" not in globals():
    TARGET_COUNT = len(YEAR_RANGES) * 150

def extract_search_links(page_source, w_param):
    """Daum 검색 결과 HTML에서 게시글 링크 목록 추출"""
    if not page_source:
        return []
    check_soup = bs(page_source, "lxml")
    if w_param == "fusion":
        links = [el.get("data-href") for el in check_soup.select("c-title[data-href]") if el.get("data-href")]
    else:
        links = [el.get("href") for el in check_soup.select(".c-list-basic .item-title a, .tit-g a") if el.get("href")]
    return links

ZERO_LOCK = threading.Lock()
WARN_LOCK = threading.Lock()
WARN_FILE = "./수집_경고.txt"  # 수집 경고 기록 (노트북 파일과 같은 폴더에 저장. 어떤 파일의 어느 부분인지)
if "POST_WARN" not in globals():
    POST_WARN = {}  # 정규화 URL -> 경고 문구 (캐시에서 복사된 글에도 같은 경고를 기록하기 위함)
if "POST_FAIL_LOGGED" not in globals():
    POST_FAIL_LOGGED = set()  # 글 열기/추출에 실패해 경고를 이미 기록한 URL (같은 글을 키워드마다 중복 기록하지 않기 위함)
SEARCH_PAGE_SIZE = 15  # 다음 fusion 검색은 한 페이지에 15개: 15개보다 적게 온 페이지는 마지막 페이지이므로 다음 페이지를 요청하지 않음
COLUMNS = ["date", "keyword", "title", "contents", "comments", "site", "url", "author_nickname", "comment_nicknames"]

# 진행 표시용 전체 집계 (워커들이 함께 올림)
STATS = {"posts": 0, "cache_hits": 0, "failed_posts": 0, "blacklisted": 0, "task_errors": 0}
_stats_lock = threading.Lock()


def bump(**kw):
    with _stats_lock:
        for k, v in kw.items():
            STATS[k] += v

def crawl_task(target_id, key, disp_handle=None):
    """단일 (수집대상, 키워드) 조합을 월별 구간 및 관련도순으로 크롤링하는 작업 함수

    글 단위 문제는 그 글만 건너뛰고 경고(수집_경고.txt)로 기록합니다.
      - 브런치/티스토리 댓글 수집 문제: 해당 글의 행 번호와 사유 기록
      - 글을 열지 못했거나 제목/본문을 읽지 못한 경우: 해당 글 URL과 사유 기록 (같은 글은 세션당 한 번만)
    검색 페이지 요청이 재시도 후에도 실패하거나 '결과 없음' 안내가 없는 빈 응답이 오면(차단/구조 변경 의심)
    그 키워드는 바로 멈추고 저장하지 않습니다(FetchError). 일부만 담긴 파일이 완료로 남지 않게 하기 위함이며, 재실행 시 다시 수집합니다.
    수집 중단 요청(STOP)이 있으면 저장하지 않고 Stopped를 발생시킵니다.
    """
    target_info = DAUM_TARGETS[target_id]
    target_name = target_info["name"]
    w_param = target_info["w"]
    col_param = target_info["col"]

    rows = []
    task_warnings = []
    search_issue = False
    seen_urls = set()
    failed_count = 0
    cache_hits = 0
    blacklisted_count = 0

    def report(msg=None):
        if disp_handle:
            disp_handle.update(msg or f"[{target_name} | {key}] 현재 수집: {len(rows)}개 (캐시 복사 {cache_hits}개), 실패: {failed_count}개, 제외(길이 초과): {blacklisted_count}개")

    report()

    encoded_key = urllib.parse.quote(key.replace("+%2B", "+"))

    try:
        for start_date_str, end_date_str in YEAR_RANGES:
            if len(rows) >= TARGET_COUNT or search_issue:
                break

            breaker = False
            page_num = 1
            max_page_limit = 11 if w_param == "fusion" else 100

            while not breaker:
                if STOP.is_set():
                    raise Stopped()
                if len(rows) >= TARGET_COUNT or page_num > max_page_limit:
                    breaker = True
                    break

                if w_param == "fusion":
                    search_url = f"https://search.daum.net/search?w=fusion&col={col_param}&q={encoded_key}&sort=accuracy&period=u&sd={start_date_str}000000&ed={end_date_str}235959&p={page_num}"
                else:
                    search_url = f"https://search.daum.net/search?w={w_param}&q={encoded_key}&sort=accuracy&period=u&sd={start_date_str}000000&ed={end_date_str}235959&p={page_num}"

                try:
                    search_html = fetch_search_page(search_url)
                except FetchError as e:
                    # 재시도 후에도 실패하면(차단/서버 오류) 이 키워드는 여기서 멈추고 저장하지 않습니다
                    task_warnings.append((f"검색 {start_date_str}~{end_date_str} p{page_num}", search_url, f"검색 요청 실패: {e}"))
                    search_issue = True
                    break
                time.sleep(random.uniform(0.25, 0.75))

                href_list = extract_search_links(search_html, w_param)

                if len(href_list) == 0:
                    if not is_confirmed_empty(search_html):
                        # '결과 없음' 안내가 확인되지 않은 빈 페이지(차단/오류/구조 변경 의심): 이 키워드는 저장하지 않음
                        task_warnings.append((f"검색 {start_date_str}~{end_date_str} p{page_num}", search_url, "결과 없음 안내를 확인하지 못한 빈 응답(차단/구조 변경 의심)"))
                        search_issue = True
                    breaker = True
                    break

                for href_value in href_list:
                    if len(rows) >= TARGET_COUNT:
                        breaker = True
                        break

                    norm = href_value  # normalize_url 이전에 오류가 나도 except에서 쓸 수 있도록 기본값 지정
                    try:
                        if any(element in href_value for element in eliminate_cafe):
                            continue

                        norm = normalize_url(href_value)
                        if norm in existing_urls or norm in seen_urls:
                            continue
                        if norm in BLACKLIST:
                            blacklisted_count += 1
                            bump(blacklisted=1)
                            continue

                        post_data = cache_get(norm)
                        if post_data is not None:
                            cache_hits += 1
                            bump(cache_hits=1)
                        else:
                            if is_dead(href_value):
                                # 이미 삭제된 글(404/410)로 확인된 URL은 다시 요청하지 않음 (경고는 처음 한 번만 기록됨)
                                failed_count += 1
                                bump(failed_posts=1)
                                report()
                                continue
                            raw_html = fetch_html(href_value, referer=search_url)
                            time.sleep(random.uniform(0.25, 0.75))

                            post_html = resolve_post_html(href_value, raw_html, target_id=target_id)
                            post_data = extract_post_data(post_html, target_id=target_id) if post_html else None

                            if not (post_data and post_data["title"]):
                                failed_count += 1
                                bump(failed_posts=1)
                                if norm not in POST_FAIL_LOGGED:
                                    POST_FAIL_LOGGED.add(norm)
                                    if raw_html is None:
                                        fail_reason = "글 페이지 요청 실패(삭제/차단/서버 오류)"
                                    elif not post_html:
                                        fail_reason = "글 본문 프레임을 불러오지 못함"
                                    else:
                                        fail_reason = "제목/본문 추출 실패(비공개 글, 페이지 구조 다름 등)"
                                    task_warnings.append(("글 수집 실패(행 없음)", href_value, fail_reason))
                                report()
                                continue

                            if MAX_CONTENT_CHARS and len(post_data["contents"]) > MAX_CONTENT_CHARS:
                                # 본문이 너무 긴 글은 저장하지 않고 블랙리스트에 올림 (댓글 요청도 생략)
                                add_blacklist(norm, len(post_data["contents"]))
                                blacklisted_count += 1
                                bump(blacklisted=1)
                                report()
                                continue

                            if target_id in (1, 2):
                                # 브런치/티스토리 댓글은 HTML에 없고 별도 API로만 받을 수 있음 (실패해도 멈추지 않고 경고로 기록)
                                fetch_comments = fetch_brunch_comments if target_id == 2 else fetch_tistory_comments
                                post_data["comments"], post_data["comment_nicknames"], comment_warn = fetch_comments(post_html, href_value)
                                if comment_warn:
                                    POST_WARN[norm] = comment_warn
                            cache_put(norm, post_data)
                            bump(posts=1)

                        if MAX_CONTENT_CHARS and len(post_data["contents"]) > MAX_CONTENT_CHARS:
                            add_blacklist(norm, len(post_data["contents"]))  # 캐시에서 복사한 글도 같은 기준 적용
                            blacklisted_count += 1
                            bump(blacklisted=1)
                            report()
                            continue

                        seen_urls.add(norm)
                        rows.append([
                            post_data["date"],
                            key,
                            post_data["title"],
                            post_data["contents"],
                            post_data["comments"],
                            target_name,
                            href_value,
                            post_data["author_nickname"],
                            post_data["comment_nicknames"],
                        ])
                        if norm in POST_WARN:
                            task_warnings.append((f"{len(rows) + 1}행", href_value, POST_WARN[norm]))  # CSV 행 번호(헤더가 1행)
                        report()

                    except Exception as e:
                        failed_count += 1
                        bump(failed_posts=1)
                        if norm not in POST_FAIL_LOGGED:
                            POST_FAIL_LOGGED.add(norm)
                            task_warnings.append(("글 처리 오류(행 없음)", href_value, f"{type(e).__name__}: {e}"))
                        report()
                        continue

                if w_param == "fusion" and len(href_list) < SEARCH_PAGE_SIZE:
                    break  # 15개보다 적게 온 페이지가 마지막 페이지(직접 확인): 끝 확인용 요청을 생략

                # 다음 fusion 검색은 p=2가 p=1과 같은 결과를 돌려주므로(직접 확인), 1페이지 다음은 p=3으로 건너뜀
                page_num += 2 if (page_num == 1 and w_param == "fusion") else 1
                time.sleep(random.uniform(0.25, 0.75))

    finally:
        report()

    safe_keyword = key.replace("+%2B", "_").replace(" ", "").replace("/", "_")

    if search_issue:
        # 검색이 막힌 키워드는 저장하지 않습니다(일부만 담긴 파일이 완료로 남지 않도록). 경고만 남기고 재실행 시 다시 수집
        with WARN_LOCK:
            with open(WARN_FILE, "a", encoding="utf-8") as f:
                for location, warn_url, warn_msg in task_warnings:
                    f.write(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] 다음포탈_{target_name}_{safe_keyword} (저장 안 함) | {location} | {warn_url} | {warn_msg}\n")
        report(f"[{target_name} | {key}] 검색 요청 실패로 저장하지 않음 (재실행 시 다시 수집)")
        raise FetchError(f"검색 요청 실패로 저장하지 않습니다(재실행 필요): {target_name} | {key}")

    dataframe = pd.DataFrame(rows, columns=COLUMNS)
    csv_filename = f"{SAVE_PATH}다음포탈_{target_name}_{safe_keyword}_{len(dataframe)}개.csv"

    # 임시 파일에 다 쓴 뒤 바꿔치기: 쓰는 도중 멈춰도 반쯤 쓴 CSV가 '완료'로 남지 않음
    dataframe.to_csv(csv_filename + ".tmp", encoding="utf-8-sig", index=False)
    os.replace(csv_filename + ".tmp", csv_filename)
    mark_imported(csv_filename)  # 이 파일의 글은 이미 캐시에 들어 있으므로 다시 읽지 않도록 표시

    if task_warnings:
        with WARN_LOCK:
            with open(WARN_FILE, "a", encoding="utf-8") as f:
                for location, warn_url, warn_msg in task_warnings:
                    f.write(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {os.path.basename(csv_filename)} | {location} | {warn_url} | {warn_msg}\n")

    if len(dataframe) == 0:
        # 여기까지 왔다면 검색 요청이 모두 성공했고 빈 페이지가 모두 '결과 없음' 안내로 확인된 경우이므로 검증된 0건으로 기록
        with ZERO_LOCK:
            with open(VERIFIED_ZERO_FILE, "a", encoding="utf-8") as f:
                f.write(f"{target_name}\t{safe_keyword}\n")
    else:
        # 이전 실행에서 남은 검증 안 된 0개 파일은 이번 수집 결과로 대체
        stale = f"{SAVE_PATH}다음포탈_{target_name}_{safe_keyword}_0개.csv"
        if os.path.exists(stale) and os.path.getsize(stale) < 1024:
            os.remove(stale)
    report(f"[{target_name} | {key}] 수집 완료: 총 {len(dataframe)}개 저장{' (검증된 0건)' if len(dataframe) == 0 else ''} (캐시 복사 {cache_hits}개), 실패: {failed_count}개, 제외(길이 초과): {blacklisted_count}개")
    return csv_filename

# ========== 다중배치 실행 큐 구성 (이어하기 스킵 반영) ==========
tasks = []
skipped_count = 0
active_completed_keys = globals().get("completed_task_keys", set()) if globals().get("RESUME_FROM_SAVED", False) else set()

print("========== 수집 실행 대상 최종 확인 ==========")
for target_id in SELECTED_TARGETS:
    if target_id in DAUM_TARGETS:
        target_name = DAUM_TARGETS[target_id]["name"]
        target_remaining = 0
        target_skipped = 0
        for key in keywords:
            safe_kw = key.replace("+%2B", "_").replace(" ", "").replace("/", "_")
            if (target_name, safe_kw) in active_completed_keys:
                skipped_count += 1
                target_skipped += 1
                continue
            tasks.append((target_id, key))
            target_remaining += 1
        print(f"수집 대상 [{target_id}]: {target_name} (전체 {len(keywords)}개 중 기존 완료 {target_skipped}개 스킵 -> 남은 {target_remaining}개 실행)")

print(f"총 {len(tasks)}개 미수집 작업을 최대 {MAX_WORKERS}개 다중배치로 실행합니다.\n")

completed_files = []
done_count = 0
total_tasks = len(tasks)

# ---------- 진행 표시: 키워드마다 줄을 만들지 않고, 같은 자리에서 2초마다 갱신되는 요약 한 칸만 씁니다 ----------
from collections import deque
from concurrent.futures import wait, FIRST_COMPLETED
from IPython.display import HTML


class _NoHandle:
    """주피터가 아닌 환경(터미널 등)에서는 같은 자리 갱신이 없으므로 아무것도 하지 않습니다"""
    def update(self, obj):
        pass


def live(obj):
    handle = display(obj, display_id=True)
    return handle if handle is not None else _NoHandle()


def progress_html(title, done, total, extra=""):
    pct = 100 * done / total if total else 100
    return HTML(
        f"<div style='font-family:monospace;line-height:1.7'><b>{title}</b> {done:,}/{total:,} ({pct:.1f}%) "
        f"<progress value='{done}' max='{max(total, 1)}' style='width:320px;vertical-align:middle'></progress><br>"
        f"<span style='color:#555'>{extra}</span></div>")


run_targets = [t for t in SELECTED_TARGETS if t in DAUM_TARGETS]
done_by_target = {t: 0 for t in run_targets}
_started = time.time()
_rate_samples = deque([(time.time(), 0)])
_folder_cache = {"t": 0.0, "v": {t: 0 for t in run_targets}}


def folder_counts():
    """data_csv에 실제로 저장된 키워드 파일 수(대상별). 이전 실행분 포함, 5초마다 다시 셉니다"""
    if time.time() - _folder_cache["t"] > 5:
        names = os.listdir(SAVE_PATH)
        counts = {t: sum(1 for f in names if f.startswith(f"다음포탈_{DAUM_TARGETS[t]['name']}_") and f.endswith(".csv")) for t in run_targets}
        _folder_cache.update(t=time.time(), v=counts)
    return _folder_cache["v"]


def recent_rate(total):
    """최근 5분 동안의 글/분 (시작부터의 누적 평균이 아님)"""
    now = time.time()
    _rate_samples.append((now, total))
    while len(_rate_samples) > 1 and now - _rate_samples[0][0] > 300:
        _rate_samples.popleft()
    t0, n0 = _rate_samples[0]
    return (total - n0) / ((now - t0) / 60) if now - t0 >= 10 else None


def status_text():
    with _stats_lock:
        s = dict(STATS)
    el = time.time() - _started
    files = folder_counts()
    on_disk = " · ".join(f"{DAUM_TARGETS[t]['name']} {files[t]:,}/{len(keywords):,}" for t in run_targets)
    this_run = " · ".join(f"{DAUM_TARGETS[t]['name']} +{done_by_target[t]:,}" for t in run_targets)
    rate = recent_rate(s["posts"] + s["cache_hits"])
    return (f"data_csv 저장 파일: {on_disk} (이번 실행에서 {this_run})<br>"
            f"처리 중인 키워드: {min(MAX_WORKERS, total_tasks - done_count)}개<br>"
            f"새 글 {s['posts']:,} · 캐시 복사 {s['cache_hits']:,} · 글 실패 {s['failed_posts']:,} · 제외(길이 초과) {s['blacklisted']:,} · "
            f"작업 실패 {s['task_errors']:,} · {'측정 중' if rate is None else f'최근 5분 {rate:.0f}글/분'} · "
            f"네트워크 {'정상' if _network_ok.is_set() else '끊김(대기 중)'} · 경과 {int(el // 3600)}시간 {int(el % 3600 // 60)}분")


STOP.clear()
view = live(progress_html("수집", 0, total_tasks, status_text()))
executor = ThreadPoolExecutor(max_workers=MAX_WORKERS)
futures = {executor.submit(crawl_task, t_id, kw): (t_id, kw) for t_id, kw in tasks}
pending = set(futures)
try:
    # 화면 갱신은 노트북 셀을 실행하는 메인 스레드에서만 합니다(2초마다, 키워드가 끝날 때마다)
    while pending:
        done, pending = wait(pending, timeout=2, return_when=FIRST_COMPLETED)
        if done:
            _folder_cache["t"] = 0.0  # 키워드가 끝나면 폴더 파일 수를 바로 다시 셉니다
        for future in done:
            t_id, kw = futures[future]
            target_name = DAUM_TARGETS[t_id]["name"]
            done_count += 1
            done_by_target[t_id] += 1
            try:
                res_file = future.result()
                if res_file:
                    completed_files.append(res_file)
            except Exception as e:
                bump(task_errors=1)
                logging.error(f"[{done_count}/{total_tasks} 에러] 작업 실패 ({target_name} | {kw}): {e}")
        view.update(progress_html("수집", done_count, total_tasks, status_text()))
except KeyboardInterrupt:
    # Interrupt는 메인 스레드에만 전달되므로, 워커에 멈춤 신호를 주고 아직 시작하지 않은 작업을 취소합니다
    STOP.set()
    print(f"[{time.strftime('%H:%M:%S')}] 중단 요청 - 남은 작업을 취소하고, 진행 중인 키워드가 멈추길 기다립니다(보통 몇 초, 최대 10초 정도)...")
    executor.shutdown(wait=False, cancel_futures=True)
executor.shutdown(wait=True)
view.update(progress_html("수집", done_count, total_tasks, status_text()))
if os.path.exists(WARN_FILE):
    print(f"\n[참고] 수집 경고가 있습니다. 파일/행/URL은 {WARN_FILE} 를 확인하세요.")
if STOP.is_set():
    print(f"\n수집을 중단했습니다: 이번 세션 {len(completed_files)}개 파일 저장. 중간에 멈춘 키워드는 저장하지 않았으며, "
          f"4-2(이어하기·캐시) 셀부터 다시 실행하면 이어서 수집합니다.")
else:
    print(f"\n전체 크롤링 완료: 이번 세션 총 {total_tasks}개 중 {len(completed_files)}개 파일 정상 저장됨 (기존 스킵: {skipped_count}개)")


========== 수집 실행 대상 최종 확인 ==========
수집 대상 [0]: 다음카페 (전체 1680개 중 기존 완료 0개 스킵 -> 남은 1680개 실행)
총 1680개 미수집 작업을 최대 8개 다중배치로 실행합니다.



[20:14:24] 중단 요청 - 남은 작업을 취소하고, 진행 중인 키워드가 멈추길 기다립니다(보통 몇 초, 최대 10초 정도)...

[참고] 수집 경고가 있습니다. 파일/행/URL은 ./수집_경고.txt 를 확인하세요.

수집을 중단했습니다: 이번 세션 10개 파일 저장. 중간에 멈춘 키워드는 저장하지 않았으며, 4-2(이어하기·캐시) 셀부터 다시 실행하면 이어서 수집합니다.


## 6. 전체 데이터 통합 파일 생성

- 수집 완료된 개별 CSV 파일들을 하나로 병합하여 중복을 제거한 `TOTAL` 파일을 생성합니다.
- 기존 개별 파일들은 그대로 유지되며, 통합 파일만 새로 추가됩니다.
- 크롤링이 모두 끝난 후 실행하거나, 나중에 단독으로 실행해도 됩니다.

In [12]:
# # ========== 수집 완료된 개별 CSV 파일 통합 병합 ==========
# import glob
# import os
# import pandas as pd
#
# # TOTAL 파일을 제외한 개별 CSV 파일들 탐색
# csv_files = sorted([f for f in glob.glob(f"{SAVE_PATH}다음포탈_*.csv") if "_TOTAL_" not in f])
#
# if csv_files:
#     print(f"총 {len(csv_files)}개 개별 CSV 파일 통합 병합 시작...")
#     df_list = []
#     for f_path in csv_files:
#         try:
#             df_item = pd.read_csv(f_path, encoding="utf-8-sig")
#             df_list.append(df_item)
#             print(f"  - 로드 완료: {os.path.basename(f_path)} ({len(df_item)}행)")
#         except Exception as e:
#             print(f"  - 읽기 실패 ({os.path.basename(f_path)}): {e}")
#
#     if df_list:
#         total_df = pd.concat(df_list, ignore_index=True)
#
#         # URL 기준 중복 제거
#         if "normalize_url" in globals():
#             total_df["norm_url"] = total_df["url"].apply(normalize_url)
#             total_df = total_df.drop_duplicates(subset="norm_url", keep="first").drop(columns=["norm_url"]).reset_index(drop=True)
#         else:
#             total_df = total_df.drop_duplicates(subset="url", keep="first").reset_index(drop=True)
#
#         total_csv_filename = f"{SAVE_PATH}다음포탈_TOTAL_{len(total_df)}개.csv"
#         total_df.to_csv(total_csv_filename, encoding="utf-8-sig", index=False)
#         print(f"\n통합 파일 생성 완료: {total_csv_filename} (총 {len(total_df)}개 적재)\n")
# else:
#     print("[경고] 통합할 개별 CSV 파일이 존재하지 않습니다.")
